In [1]:
import pandas as pd
from collections import Counter, defaultdict
import numpy as np
import re, os, glob
import ast
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from ipywidgets import VBox, HBox, Button, Label, Output, Layout, Dropdown, HTML
from IPython.display import display, Markdown
from scipy.sparse import csr_matrix

LOAD kaggle dataset

In [2]:
movies  = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

movies = movies.merge(credits, left_on="id", right_on="movie_id")

movies["title"] = movies["title_x"]
movies = movies.drop(columns = ['title_x', 'title_y'])

def parse_names(x, key="name"):
    try:
        data = ast.literal_eval(x) 
    except (ValueError, SyntaxError, TypeError):
        return []
    names = []
    for d in data:
        if isinstance(d, dict) and key in d:
            names.append(d[key].replace(" ", "_").lower())
    return names

movies["genres_clean"]   = movies["genres"].apply(parse_names)
movies["keywords_clean"] = movies["keywords"].apply(parse_names)
movies["cast_clean"] = movies["cast"].apply(lambda x: parse_names(x)[:5])

def get_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
    except (ValueError, SyntaxError, TypeError):
        return []
    for d in crew:
        if isinstance(d, dict) and d.get("job") == "Director":
            return [d["name"].replace(" ", "_").lower()]
    return []

movies["director_clean"] = movies["crew"].apply(get_director)

movies["overview_clean"] = movies["overview"].fillna("").str.lower()

def make_soup(row):
    genres    = row["genres_clean"]
    keywords  = row["keywords_clean"]
    cast      = row["cast_clean"]
    director  = row["director_clean"]
    overview  = row["overview_clean"].split()[:50] 

    return " ".join(genres + keywords + cast + director + overview)

movies["soup"] = movies.apply(make_soup, axis=1)


In [3]:

movies["title_norm"] = movies["title"].str.lower().str.strip()
movies["year"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.year

movies = movies.reset_index().rename(columns={"index": "movie_idx"})

movies[["movie_idx", "title", "year"]].head()


,movie_idx,title,year
0,0,Avatar,2009.0
1,1,Pirates of the Caribbean: At World's End,2007.0
2,2,Spectre,2015.0
3,3,The Dark Knight Rises,2012.0
4,4,John Carter,2012.0


LOAD LETTERBOXD DATA

In [4]:
# TF–IDF over the soup
tfidf = TfidfVectorizer(stop_words="english", min_df=2)
X = tfidf.fit_transform(movies["soup"])

X.shape


(4803, 16741)

In [5]:

lb_dirs = glob.glob("letterboxd-*")

all_ratings = []

for folder in lb_dirs:
    user_id = os.path.basename(folder).replace("letterboxd-", "")
    ratings_path = os.path.join(folder, "ratings.csv")
    
    if not os.path.exists(ratings_path):
        continue
    
    df = pd.read_csv(ratings_path)
    df["user_id"] = user_id          # tag who this rating belongs to
    all_ratings.append(df)

lb_ratings_all = pd.concat(all_ratings, ignore_index=True) if all_ratings else pd.DataFrame()

print("Ratings shape:", lb_ratings_all.shape)
lb_ratings_all.head()


Ratings shape: (1170, 6)


,Date,Name,Year,Letterboxd URI,Rating,user_id
0,2023-02-07,Spider-Man: Into the Spider-Verse,2018,https://boxd.it/azpY,4.0,gabybrown
1,2023-02-07,Puss in Boots: The Last Wish,2022,https://boxd.it/aaie,4.0,gabybrown
2,2023-02-07,Triangle of Sadness,2022,https://boxd.it/hXlq,5.0,gabybrown
3,2023-02-07,How to Lose a Guy in 10 Days,2003,https://boxd.it/1XwG,4.0,gabybrown
4,2023-02-07,Mike and Dave Need Wedding Dates,2016,https://boxd.it/aciK,5.0,gabybrown


In [6]:
# favourites (seperate top 4 into different rows)

lb_dirs = glob.glob("letterboxd-*")

all_favorites = []

def normalize_lb_url(u: str) -> str:
    """Make Letterboxd URLs comparable between profile and ratings."""
    u = str(u).strip()
    u = u.replace("https://letterboxd.com", "")
    u = u.replace("http://letterboxd.com", "")
    return u

for folder in lb_dirs:
    user_id = os.path.basename(folder).replace("letterboxd-", "")
    profile_path = os.path.join(folder, "profile.csv")
    ratings_path = os.path.join(folder, "ratings.csv")
    
    if not os.path.exists(profile_path) or not os.path.exists(ratings_path):
        continue
    
    prof = pd.read_csv(profile_path)
    if prof.empty:
        continue
    
    fav_cols = [c for c in prof.columns if "favorite" in c.lower()]
    if not fav_cols:
        continue
    
    row = prof.iloc[0]
    fav_series = row[fav_cols]
    
    urls = fav_series.dropna().astype(str).tolist()
    if not urls:
        continue
    
    fav_df = pd.DataFrame({
        "user_id": user_id,
        "lb_url": urls,
        "fav_rank": range(1, len(urls) + 1),   
    })
    
    ratings_df = pd.read_csv(ratings_path)
    
    uri_cols = [c for c in ratings_df.columns if "uri" in c.lower()]
    if not uri_cols:
        all_favorites.append(fav_df)
        continue
    
    uri_col = uri_cols[0]
    
    ratings_df["lb_url"] = ratings_df[uri_col].astype(str).apply(normalize_lb_url)
    fav_df["lb_url"] = fav_df["lb_url"].astype(str).apply(normalize_lb_url)
    
    fav_with_titles = fav_df.merge(
        ratings_df,
        on="lb_url",
        how="left",
        suffixes=("", "_rating")
    )
    
    all_favorites.append(fav_with_titles)

user_favorites_all = pd.concat(all_favorites, ignore_index=True) if all_favorites else pd.DataFrame()

print("Favorites shape:", user_favorites_all.shape)
user_favorites_all.head()


Favorites shape: (5, 8)


,user_id,lb_url,fav_rank,Date,Name,Year,Letterboxd URI,Rating
0,gabybrown,"https://boxd.it/2bcU, https://boxd.it/2aSg, ht...",1,NaN,NaN,NaN,NaN,NaN
1,bobblet11,"https://boxd.it/20Z2, https://boxd.it/2aHi, ht...",1,NaN,NaN,NaN,NaN,NaN
2,ariannagk,"https://boxd.it/29VS, https://boxd.it/27V2, ht...",1,NaN,NaN,NaN,NaN,NaN
3,tharaaaaa,"https://boxd.it/dpik, https://boxd.it/3OsE, ht...",1,NaN,NaN,NaN,NaN,NaN
4,megwhat,"https://boxd.it/a5fa, https://boxd.it/iEBG, ht...",1,NaN,NaN,NaN,NaN,NaN


In [7]:
user_favorites_all["lb_url_list"] = user_favorites_all["lb_url"].astype(str).str.split(",")

user_favorites_all = (
    user_favorites_all
    .explode("lb_url_list")
    .drop(columns=["lb_url"])          
    .rename(columns={"lb_url_list": "lb_url"})
)

user_favorites_all["lb_url"] = user_favorites_all["lb_url"].str.strip()

user_favorites_all = (
    user_favorites_all
    .sort_values(["user_id", "lb_url"])  
    .reset_index(drop=True)
)
user_favorites_all["fav_rank"] = (
    user_favorites_all
    .groupby("user_id")
    .cumcount() + 1
)

print("Favorites shape after explode:", user_favorites_all.shape)
user_favorites_all


Favorites shape after explode: (20, 8)


,user_id,fav_rank,Date,Name,Year,Letterboxd URI,Rating,lb_url
0,ariannagk,1,NaN,NaN,NaN,NaN,NaN,https://boxd.it/18Im
1,ariannagk,2,NaN,NaN,NaN,NaN,NaN,https://boxd.it/27V2
2,ariannagk,3,NaN,NaN,NaN,NaN,NaN,https://boxd.it/29VS
3,ariannagk,4,NaN,NaN,NaN,NaN,NaN,https://boxd.it/2asW
4,bobblet11,1,NaN,NaN,NaN,NaN,NaN,https://boxd.it/20Z2
5,bobblet11,2,NaN,NaN,NaN,NaN,NaN,https://boxd.it/293w
6,bobblet11,3,NaN,NaN,NaN,NaN,NaN,https://boxd.it/29FA
7,bobblet11,4,NaN,NaN,NaN,NaN,NaN,https://boxd.it/2aHi
8,gabybrown,1,NaN,NaN,NaN,NaN,NaN,https://boxd.it/2aOe
9,gabybrown,2,NaN,NaN,NaN,NaN,NaN,https://boxd.it/2aSg


In [8]:
# merge favourites with ratings df 
lb_ratings_all["Letterboxd URI"] = lb_ratings_all["Letterboxd URI"].astype(str).str.strip()
user_favorites_all["lb_url"] = user_favorites_all["lb_url"].astype(str).str.strip()

cols_to_drop = ["Name", "Year", "Rating", "Date", "Letterboxd URI"]
user_favorites_all = user_favorites_all.drop(columns=cols_to_drop, errors="ignore")

ratings_small = lb_ratings_all[[
    "user_id",
    "Letterboxd URI",
    "Name",
    "Year",
    "Rating",
    "Date",
]].copy()

ratings_small = ratings_small.rename(columns={"Letterboxd URI": "lb_url"})

user_favorites_all = user_favorites_all.merge(
    ratings_small,
    on=["user_id", "lb_url"],
    how="left"
)

user_favorites_all["fav_rank"] = 1

print("Favorites shape after merge:", user_favorites_all.shape)
user_favorites_all[["user_id", "lb_url", "fav_rank", "Name", "Year", "Rating", "Date"]]


Favorites shape after merge: (20, 7)


,user_id,lb_url,fav_rank,Name,Year,Rating,Date
0,ariannagk,https://boxd.it/18Im,1,The Graduate,1967.0,5.0,2025-07-12
1,ariannagk,https://boxd.it/27V2,1,Star Wars: Episode II – Attack of the Clones,2002.0,5.0,2024-03-11
2,ariannagk,https://boxd.it/29VS,1,Catch Me If You Can,2002.0,5.0,2024-03-11
3,ariannagk,https://boxd.it/2asW,1,Garden State,2004.0,4.5,2024-10-17
4,bobblet11,https://boxd.it/20Z2,1,There Will Be Blood,2007.0,4.0,2025-09-13
5,bobblet11,https://boxd.it/293w,1,NaN,NaN,NaN,NaN
6,bobblet11,https://boxd.it/29FA,1,NaN,NaN,NaN,NaN
7,bobblet11,https://boxd.it/2aHi,1,NaN,NaN,NaN,NaN
8,gabybrown,https://boxd.it/2aOe,1,Stand by Me,1986.0,5.0,2023-09-26
9,gabybrown,https://boxd.it/2aSg,1,Dead Poets Society,1989.0,5.0,2023-08-06


In [9]:
manual_titles = {
    "https://boxd.it/293w": "The Prestige",
    "https://boxd.it/29FA": "Goodfellas",
    "https://boxd.it/2aHi": "The Shawshank Redemption",
}

user_favorites_all["Name"] = user_favorites_all["lb_url"].map(manual_titles).fillna(user_favorites_all["Name"])

mask = user_favorites_all["lb_url"].isin(manual_titles.keys())
user_favorites_all.loc[mask, "Rating"] = 5.0

user_favorites_all


,user_id,fav_rank,lb_url,Name,Year,Rating,Date
0,ariannagk,1,https://boxd.it/18Im,The Graduate,1967.0,5.0,2025-07-12
1,ariannagk,1,https://boxd.it/27V2,Star Wars: Episode II – Attack of the Clones,2002.0,5.0,2024-03-11
2,ariannagk,1,https://boxd.it/29VS,Catch Me If You Can,2002.0,5.0,2024-03-11
3,ariannagk,1,https://boxd.it/2asW,Garden State,2004.0,4.5,2024-10-17
4,bobblet11,1,https://boxd.it/20Z2,There Will Be Blood,2007.0,4.0,2025-09-13
5,bobblet11,1,https://boxd.it/293w,The Prestige,NaN,5.0,NaN
6,bobblet11,1,https://boxd.it/29FA,Goodfellas,NaN,5.0,NaN
7,bobblet11,1,https://boxd.it/2aHi,The Shawshank Redemption,NaN,5.0,NaN
8,gabybrown,1,https://boxd.it/2aOe,Stand by Me,1986.0,5.0,2023-09-26
9,gabybrown,1,https://boxd.it/2aSg,Dead Poets Society,1989.0,5.0,2023-08-06


In [10]:
fav_keys = (
    user_favorites_all[["user_id", "lb_url"]]
    .dropna()
    .drop_duplicates()
    .assign(favorite=1)
)


In [11]:
lb_ratings_all["lb_url"] = lb_ratings_all["Letterboxd URI"]

lb_ratings_all = lb_ratings_all.merge(
    fav_keys,
    on=["user_id", "lb_url"],
    how="left"
)

lb_ratings_all["favorite"] = lb_ratings_all["favorite"].fillna(0).astype(int)

lb_ratings_all[["user_id", "Name", "Rating", "favorite"]]


,user_id,Name,Rating,favorite
0,gabybrown,Spider-Man: Into the Spider-Verse,4.0,0
1,gabybrown,Puss in Boots: The Last Wish,4.0,0
2,gabybrown,Triangle of Sadness,5.0,0
3,gabybrown,How to Lose a Guy in 10 Days,4.0,0
4,gabybrown,Mike and Dave Need Wedding Dates,5.0,0
...,...,...,...,...
1165,megwhat,The Boy and the Heron,2.5,0
1166,megwhat,KPop Demon Hunters,3.5,0
1167,megwhat,Woman of the Hour,3.0,0
1168,megwhat,Catch Me If You Can,5.0,1


In [12]:
lb_ratings_all["favorite"].value_counts()

favorite
0    1153
1      17
Name: count, dtype: int64

In [13]:
# fake bobbi favourites for consistency, he didn't rate his favourites, it doesn't show

bobbi_id = "bobblet11" 

bobbi_manual = pd.DataFrame([
    {
        "user_id": bobbi_id,
        "Name": "The Prestige",
        "Year": 2006,
        "Rating": 5.0,
        "Date": pd.NaT,
        "Letterboxd URI": "https://boxd.it/293w",
        "lb_url": "https://boxd.it/293w",
        "favorite": 1,
    },
    {
        "user_id": bobbi_id,
        "Name": "Goodfellas",
        "Year": 1990,
        "Rating": 5.0,
        "Date": pd.NaT,
        "Letterboxd URI": "https://boxd.it/29FA",
        "lb_url": "https://boxd.it/29FA",
        "favorite": 1,
    },
    {
        "user_id": bobbi_id,
        "Name": "The Shawshank Redemption",
        "Year": 1994,
        "Rating": 5.0,
        "Date": pd.NaT,
        "Letterboxd URI": "https://boxd.it/2aHi",
        "lb_url": "https://boxd.it/2aHi",
        "favorite": 1,
    },
])

if "favorite" not in lb_ratings_all.columns:
    lb_ratings_all["favorite"] = 0

for col in lb_ratings_all.columns:
    if col not in bobbi_manual.columns:
        bobbi_manual[col] = np.nan

bobbi_manual = bobbi_manual[lb_ratings_all.columns]

lb_ratings_all = pd.concat([lb_ratings_all, bobbi_manual], ignore_index=True)

lb_ratings_all[lb_ratings_all["user_id"] == bobbi_id][["Name", "Rating", "favorite"]].tail(5)


,Name,Rating,favorite
323,Frankenstein,3.0,0
324,The Revenant,3.5,0
1170,The Prestige,5.0,1
1171,Goodfellas,5.0,1
1172,The Shawshank Redemption,5.0,1


In [14]:
lb_ratings_all[lb_ratings_all["favorite"] == 1]

,Date,Name,Year,Letterboxd URI,Rating,user_id,lb_url,favorite
33,2023-03-17,La La Land,2016,https://boxd.it/a5fa,5.0,gabybrown,https://boxd.it/a5fa,1
51,2023-08-06,Dead Poets Society,1989,https://boxd.it/2aSg,5.0,gabybrown,https://boxd.it/2aSg,1
59,2023-09-06,Before Sunrise,1995,https://boxd.it/2bcU,5.0,gabybrown,https://boxd.it/2bcU,1
63,2023-09-26,Stand by Me,1986,https://boxd.it/2aOe,5.0,gabybrown,https://boxd.it/2aOe,1
305,2025-09-13,There Will Be Blood,2007,https://boxd.it/20Z2,4.0,bobblet11,https://boxd.it/20Z2,1
325,2024-03-11,Catch Me If You Can,2002,https://boxd.it/29VS,5.0,ariannagk,https://boxd.it/29VS,1
327,2024-03-11,Star Wars: Episode II – Attack of the Clones,2002,https://boxd.it/27V2,5.0,ariannagk,https://boxd.it/27V2,1
610,2024-10-17,Garden State,2004,https://boxd.it/2asW,4.5,ariannagk,https://boxd.it/2asW,1
794,2025-07-12,The Graduate,1967,https://boxd.it/18Im,5.0,ariannagk,https://boxd.it/18Im,1
924,2023-09-29,Top Gun: Maverick,2022,https://boxd.it/cjr4,5.0,tharaaaaa,https://boxd.it/cjr4,1


In [15]:
# manual friends mapping, because I have no access to friends with current data

users = sorted(lb_ratings_all["user_id"].unique().tolist())

bob = "bobblet11"
gaby = "gabybrown"

friends_rows = []
for u in users:
    if u == bob:
        # bobblet11 only friends with gaby
        friends_list = [gaby]
    elif u == gaby:
        # gaby is friends with everyone else (including bob)
        friends_list = [v for v in users if v != gaby]
    else:
        # everyone else: friends with everyone except themselves and bob
        friends_list = [v for v in users if v not in {u, bob}]
    friends_rows.append({"user_id": u, "friends": friends_list})

user_friends_list = pd.DataFrame(friends_rows)

friends_lookup = {
    row["user_id"]: row["friends"]
    for _, row in user_friends_list.iterrows()
}

user_friends_list 


,user_id,friends
0,ariannagk,"[gabybrown, megwhat, tharaaaaa]"
1,bobblet11,[gabybrown]
2,gabybrown,"[ariannagk, bobblet11, megwhat, tharaaaaa]"
3,megwhat,"[ariannagk, gabybrown, tharaaaaa]"
4,tharaaaaa,"[ariannagk, gabybrown, megwhat]"


In [16]:
lb_ratings_all[
    lb_ratings_all["Year"] >= 2015
]

,Date,Name,Year,Letterboxd URI,Rating,user_id,lb_url,favorite
0,2023-02-07,Spider-Man: Into the Spider-Verse,2018,https://boxd.it/azpY,4.0,gabybrown,https://boxd.it/azpY,0
1,2023-02-07,Puss in Boots: The Last Wish,2022,https://boxd.it/aaie,4.0,gabybrown,https://boxd.it/aaie,0
2,2023-02-07,Triangle of Sadness,2022,https://boxd.it/hXlq,5.0,gabybrown,https://boxd.it/hXlq,0
4,2023-02-07,Mike and Dave Need Wedding Dates,2016,https://boxd.it/aciK,5.0,gabybrown,https://boxd.it/aciK,0
6,2023-02-07,Call Me by Your Name,2017,https://boxd.it/dYmm,4.5,gabybrown,https://boxd.it/dYmm,0
...,...,...,...,...,...,...,...,...
1164,2025-08-26,F1,2025,https://boxd.it/yjVM,3.0,megwhat,https://boxd.it/yjVM,0
1165,2025-09-05,The Boy and the Heron,2023,https://boxd.it/ipeM,2.5,megwhat,https://boxd.it/ipeM,0
1166,2025-09-26,KPop Demon Hunters,2025,https://boxd.it/ufwK,3.5,megwhat,https://boxd.it/ufwK,0
1167,2025-10-01,Woman of the Hour,2023,https://boxd.it/vuz2,3.0,megwhat,https://boxd.it/vuz2,0


In [17]:
lb_ratings_all["title_norm"] = (
    lb_ratings_all["Name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

movies_key = movies[["movie_idx", "title_norm", "year"]].copy()

lb_ratings_all = lb_ratings_all.merge(
    movies_key,
    left_on=["title_norm", "Year"],
    right_on=["title_norm", "year"],
    how="left"
)

lb_ratings_all[["user_id", "Name", "Year", "movie_idx"]].head(20)

#movies past 2015 have no index 


,user_id,Name,Year,movie_idx
0,gabybrown,Spider-Man: Into the Spider-Verse,2018,NaN
1,gabybrown,Puss in Boots: The Last Wish,2022,NaN
2,gabybrown,Triangle of Sadness,2022,NaN
3,gabybrown,How to Lose a Guy in 10 Days,2003,910.0
4,gabybrown,Mike and Dave Need Wedding Dates,2016,NaN
5,gabybrown,27 Dresses,2008,1560.0
6,gabybrown,Call Me by Your Name,2017,NaN
7,gabybrown,10 Things I Hate About You,1999,2739.0
8,gabybrown,Palm Springs,2020,NaN
9,gabybrown,Mamma Mia!,2008,886.0


In [18]:
LIKE_THRESHOLD = 4.5

strong = lb_ratings_all[
    ((lb_ratings_all["Rating"] >= LIKE_THRESHOLD) | (lb_ratings_all["favorite"] == 1))
    & lb_ratings_all["movie_idx"].notna()
].copy()

strong = strong.merge(
    movies[["movie_idx", "title", "cast_clean"]],
    on="movie_idx",
    how="left"
)

user_actor_map = {}

for uid, group in strong.groupby("user_id"):
    actor_info = {}
    for _, row in group.iterrows():
        rating = row["Rating"]
        title  = row["title"]
        for actor in row.get("cast_clean", []) or []:
            if actor not in actor_info or rating > actor_info[actor][1]:
                actor_info[actor] = (title, rating)
    user_actor_map[uid] = actor_info


In [19]:
coverage = lb_ratings_all["movie_idx"].notna().mean()
print(f"Fraction of ratings mapped to TMDB 5000: {coverage:.2%}")


Fraction of ratings mapped to TMDB 5000: 45.95%


In [20]:
def rating_to_weight(r):
    """
    Map Letterboxd rating (0-5) to a weight in [0, 1].

    - ratings <= 2.5 -> treat as not 'liked' (weight 0)
    - ratings > 2.5  -> linearly scaled: 2.5 -> 0, 5.0 -> 1
    """
    if pd.isna(r):
        return 0.0
    r = float(r)
    if r <= 2.5:
        return 0.0
    return (r - 2.5) / 2.5 


def rating_favorite_to_weight(rating, favorite):
    """
    Combine rating + favorite into one weight.
    - base from rating_to_weight
    - +0.4 boost if it's one of the user's top-4 favorites
    """
    base = rating_to_weight(rating)
    fav_boost = 0.4 if favorite == 1 else 0.0 
    return float(min(1.0, base + fav_boost))


In [21]:
def build_user_profile(user_id, ratings_df, X):
    """
    Build a content-based profile vector for one user.

    ratings_df: lb_ratings_all with columns:
        - user_id
        - movie_idx
        - Rating
        - favorite (0/1)
    X: TF-IDF matrix of shape (n_movies, n_features)

    Returns:
        1D numpy array (n_features,) or None if we can't build a profile.
    """
    df_u = ratings_df[
        (ratings_df["user_id"] == user_id) &
        (ratings_df["movie_idx"].notna())
    ].copy()

    if df_u.empty:
        return None

    df_u["weight"] = df_u.apply(
        lambda row: rating_favorite_to_weight(row["Rating"], row["favorite"]),
        axis=1
    )

    df_u = df_u[df_u["weight"] > 0]
    if df_u.empty:
        return None

    idxs = df_u["movie_idx"].astype(int).to_numpy()
    weights = df_u["weight"].to_numpy()


    M = X[idxs]  # shape (k, n_features)

    weighted_sum = M.T.dot(weights) 
    total_weight = weights.sum()
    if total_weight == 0:
        return None

    user_vec = weighted_sum / total_weight
    return np.asarray(user_vec).ravel()


In [22]:
def inspect_user_weights(user_id, ratings_df):
    """
    Show each movie this user 'likes' (rating/favorite),
    the computed weight, and how much it contributes.
    """
    df_u = ratings_df[
        (ratings_df["user_id"] == user_id) &
        (ratings_df["movie_idx"].notna())
    ].copy()

    if df_u.empty:
        print("No ratings for user:", user_id)
        return pd.DataFrame()

    df_u["weight"] = df_u.apply(
        lambda row: rating_favorite_to_weight(row["Rating"], row["favorite"]),
        axis=1
    )

    # keep only movies with positive weight
    df_u = df_u[df_u["weight"] > 0].copy()
    if df_u.empty:
        print("User has no positively weighted movies (all <= 2.5 stars).")
        return pd.DataFrame()

    total_w = df_u["weight"].sum()
    df_u["weight_frac"] = df_u["weight"] / total_w

    # attach movie titles from movies df
    m_lookup = movies.set_index("movie_idx")[["title", "year"]]
    df_u["movie_idx"] = df_u["movie_idx"].astype(int)
    df_u = df_u.merge(
        m_lookup,
        left_on="movie_idx",
        right_index=True,
        how="left"
    )

    # nicer ordering
    df_u = df_u.sort_values("weight", ascending=False)

    display_cols = ["Name", "Rating", "favorite", "weight", "weight_frac", "title"]
    return df_u[display_cols]


In [23]:
inspect_user_weights("gabybrown", lb_ratings_all)


,Name,Rating,favorite,weight,weight_frac,title
9,Mamma Mia!,5.0,0,1.0,0.013699,Mamma Mia!
16,Tangled,5.0,0,1.0,0.013699,Tangled
14,The Social Network,5.0,0,1.0,0.013699,The Social Network
12,Pitch Perfect,5.0,0,1.0,0.013699,Pitch Perfect
35,Gone Girl,5.0,0,1.0,0.013699,Gone Girl
...,...,...,...,...,...,...
178,Hocus Pocus,3.0,0,0.2,0.002740,Hocus Pocus
244,"Girl, Interrupted",3.0,0,0.2,0.002740,"Girl, Interrupted"
249,Silver Linings Playbook,3.0,0,0.2,0.002740,Silver Linings Playbook
252,Leaving Las Vegas,3.0,0,0.2,0.002740,Leaving Las Vegas


In [24]:
some_user = lb_ratings_all["user_id"].iloc[0]
user_vec = build_user_profile(some_user, lb_ratings_all, X)
user_vec.shape


(16741,)

In [25]:
def build_user_like_metadata(user_id, ratings_df, movies, min_rating_for_like=4.0):
    """
    Collect metadata about movies this user likes (favorites or high rating),
    for use in explanations.
    Returns:
        df_like  : DataFrame of liked movies joined with movies metadata
        dir_map  : dict director_token -> set of titles where that director appears
        cast_map : dict actor_token -> set of titles
        genre_map: dict genre_token -> set of titles
    """
    df_u = ratings_df[
        (ratings_df["user_id"] == user_id) &
        (ratings_df["movie_idx"].notna())
    ].copy()
    if df_u.empty:
        return pd.DataFrame(), {}, {}, {}

    df_like = df_u[
        (df_u["favorite"] == 1) | (df_u["Rating"] >= min_rating_for_like)
    ].copy()

    if df_like.empty:
        return pd.DataFrame(), {}, {}, {}

    df_like["movie_idx"] = df_like["movie_idx"].astype(int)
    df_like = df_like.merge(
        movies[
            ["movie_idx", "title", "year",
             "genres_clean", "cast_clean", "director_clean"]
        ],
        on="movie_idx",
        how="left"
    )
    dir_map = {}
    cast_map = {}
    genre_map = {}

    for _, row in df_like.iterrows():
        title = row["title"]
        for d in row.get("director_clean", []) or []:
            dir_map.setdefault(d, set()).add(title)
        for c in row.get("cast_clean", []) or []:
            cast_map.setdefault(c, set()).add(title)
        for g in row.get("genres_clean", []) or []:
            genre_map.setdefault(g, set()).add(title)

    return df_like, dir_map, cast_map, genre_map


In [26]:
def explain_match(rec_row,
                  dir_map,
                  cast_map,
                  genre_map,
                  favorite_genres,
                  liked_idxs,
                  liked_title_by_idx,
                  liked_rating_by_idx,
                  X):
    """
    Build a list of human-readable reasons for ONE recommended movie.
    Each category (director / cast / genres / closest-movie) appears at most once.
    """
    reasons = []

# director
    director_hit = None
    for d in rec_row.get("director_clean", []) or []:
        if d in dir_map:
            fav_title = sorted(dir_map[d])[0]
            d_name = d.replace("_", " ").title()
            director_hit = (fav_title, d_name)
            break   
    if director_hit:
        fav_title, d_name = director_hit
        reasons.append(f"Same director as **{fav_title}** ({d_name}).")

# cast
    cast_hits = []
    for c in rec_row.get("cast_clean", []) or []:
        if c in cast_map:
            fav_title = sorted(cast_map[c])[0]
            c_name = c.replace("_", " ").title()
            cast_hits.append((c_name, fav_title))

    if cast_hits:
        c_name, fav_title = cast_hits[0]     # only the strongest/first cast overlap
        reasons.append(
            f"Overlapping cast: **{c_name}** (also in **{fav_title}**)."
        )

# genre
    rec_genres = rec_row.get("genres_clean", []) or []
    # 3a) genres you often rate highly
    shared_fav = [g for g in rec_genres if g in favorite_genres]
    if shared_fav:
        nice = ", ".join(g.replace("_", " ").title() for g in shared_fav)
        reasons.append(f"Matches genres you often rate highly: {nice}.")
    else:
        # 3b) fallback: any overlapping genres
        shared_general = [g for g in rec_genres if g in genre_map]
        if shared_general:
            nice = ", ".join(g.replace("_", " ").title() for g in shared_general)
            reasons.append(f"Similar genres: {nice}.")

 #  Closest liked movie (only if similarity is strong enough) 
    if liked_idxs:
        mid = int(rec_row["movie_idx"])
        sims_to_likes = cosine_similarity(X[mid], X[liked_idxs]).ravel()
        best_sim = sims_to_likes.max()
        CLOSE_THRESHOLD = 0.30   # require at least this similarity

        if best_sim >= CLOSE_THRESHOLD:
            j = sims_to_likes.argmax()
            like_idx = liked_idxs[j]
            like_title = liked_title_by_idx.get(like_idx)
            like_rating = liked_rating_by_idx.get(like_idx)

            if like_title:
                if like_rating is not None:
                    reasons.append(
                        f"One of the closer matches in our catalog to **{like_title}**, "
                        f"which you rated {like_rating}."
                    )
                else:
                    reasons.append(
                        f"Story-wise fairly close to **{like_title}** in your diary."
                    )

# fallback
    if not reasons:
        reasons.append(
            "Strong content similarity to films you’ve enjoyed (plot, cast, and genres)."
        )

    return reasons


In [27]:
def friend_reasons_for_movie(user_id, movie_idx, ratings_df, friends_lookup):
    """
    Given a user and a candidate movie_idx, return a list of
    explanation strings based on what the user's friends did.
    - Only mention ratings >= 4.0
    """
    reasons = []

    friends = friends_lookup.get(user_id, [])
    if not friends:
        return reasons

    rows = ratings_df[
        (ratings_df["movie_idx"] == movie_idx) &
        (ratings_df["user_id"].isin(friends))
    ]

    if rows.empty:
        return reasons

    rows = rows.sort_values("Rating", ascending=False).head(3)

    for _, r in rows.iterrows():
        friend = r["user_id"]
        rating = r.get("Rating", None)
        fav_flag = r.get("is_favorite", 0)

        # Only mention rating if it's >= 4
        if pd.notna(rating) and rating >= 4.0:
            rating_str = f"{rating:.1f}".rstrip("0").rstrip(".")
            reasons.append(f"Your friend **{friend}** rated this {rating_str}.")

        if fav_flag == 1:
            reasons.append(f"Your friend **{friend}** has this in their Top 4.")

    return reasons


In [28]:
def friend_boost_for_movie(user_id, movie_idx, ratings_df, friends_lookup,
                           min_friend_rating=4.0, fav_bonus=0.15, rating_bonus=0.1):
    """
    Compute an extra boost to similarity if user's friends like this movie.
    """
    friends = friends_lookup.get(user_id, [])
    if not friends:
        return 0.0

    df_friends = ratings_df[
        (ratings_df["user_id"].isin(friends)) &
        (ratings_df["movie_idx"] == movie_idx)
    ]

    if df_friends.empty:
        return 0.0

    boost = 0.0
    # +0.15 if any friend has it as favorite
    if (df_friends["favorite"] == 1).any():
        boost += fav_bonus
    # +0.1 if any friend rated >= 4
    if (df_friends["Rating"] >= min_friend_rating).any():
        boost += rating_bonus

    return boost


In [29]:
def clean_and_pad_reasons(reasons_raw, min_reasons=3):
    """
    - Deduplicate friend reasons.
    - Enforce at most one reason per category.
    - Prioritise: director > cast > friend > genres > other/generic.
    - If there still aren't enough, add AT MOST ONE padding line.
    """
    cleaned = []
    seen_text = set()
    seen_friends = set()

    for r in reasons_raw:
        if not isinstance(r, str):
            continue
        r = r.strip()
        if not r:
            continue

        if r in seen_text:
            continue

        m = re.match(r"Your friend (\S+)", r)
        if m:
            friend = m.group(1)
            if friend in seen_friends:
                continue
            seen_friends.add(friend)

        seen_text.add(r)
        cleaned.append(r)

    if not cleaned:
        return ["Trust us, this sits very close to films you tend to enjoy."]

    director_reasons = []
    cast_reasons     = []
    friend_reasons   = []
    genre_reasons    = []
    generic_reasons  = []
    other_reasons    = []

    for r in cleaned:
        if r.startswith("Same director"):
            director_reasons.append(r)
        elif r.startswith("Overlapping cast"):
            cast_reasons.append(r)
        elif r.startswith("Your friend "):
            friend_reasons.append(r)
        elif r.startswith("Matches genres") or r.startswith("Similar genres"):
            genre_reasons.append(r)
        elif (
            "similarity to movies you've rated highly" in r
            or "Blends your favorite genres" in r
            or "close to films you love" in r
        ):
            generic_reasons.append(r)
        else:
            other_reasons.append(r)

    ordered = []

    if director_reasons:
        ordered.append(director_reasons[0])

    if cast_reasons and len(ordered) < min_reasons:
        ordered.append(cast_reasons[0])

    if friend_reasons and len(ordered) < min_reasons:
        ordered.append(friend_reasons[0])

    if genre_reasons and len(ordered) < min_reasons:
        ordered.append(genre_reasons[0])

    if other_reasons and len(ordered) < min_reasons:
        ordered.append(other_reasons[0])

    if generic_reasons and len(ordered) < min_reasons:
        ordered.append(generic_reasons[0])

    if len(ordered) < min_reasons:
        # fallback
        ordered.append("Trust us, this sits very close to films you tend to enjoy.")

    return ordered[:min_reasons]


In [30]:
def recommend_for_user_with_explanations(user_id,
                                         ratings_df,
                                         X,
                                         movies,
                                         user_friends=None,
                                         top_n=10):
    """
    Content-based recommendation using TF-IDF profile, with:
      - match percentages
      - explanation strings
      - optional friend-boost on scores
    """

    user_vec = build_user_profile(user_id, ratings_df, X)
    if user_vec is None:
        print("Could not build profile for user:", user_id)
        return pd.DataFrame()

    user_vec_2d = user_vec.reshape(1, -1)
    sims = cosine_similarity(user_vec_2d, X).ravel()  # (n_movies,)

    seen = set(
        ratings_df.loc[
            ratings_df["user_id"] == user_id, "movie_idx"
        ].dropna().astype(int).tolist()
    )

    candidates = []
    for i, s in enumerate(sims):
        if i in seen:
            continue
        candidates.append([i, s])

    if not candidates:
        return pd.DataFrame()

    candidates = np.array(candidates, dtype=float)
    cand_idxs = candidates[:, 0].astype(int)
    base_scores = candidates[:, 1]

    total_scores = base_scores.copy()
    if user_friends is not None:
        for k, mid in enumerate(cand_idxs):
            total_scores[k] += friend_boost_for_movie(
                user_id, mid, ratings_df, friends_lookup
            )

    order = np.argsort(-total_scores)
    cand_idxs = cand_idxs[order][:top_n]
    total_scores = total_scores[order][:top_n]
    base_scores = base_scores[order][:top_n]

    s_min, s_max = total_scores.min(), total_scores.max()
    if s_max > s_min:
        match_pct = 50 + 50 * (total_scores - s_min) / (s_max - s_min)
    else:
        match_pct = np.full_like(total_scores, 75.0)

    recs = movies.iloc[cand_idxs].copy()
    recs["movie_idx"] = cand_idxs
    recs["raw_score"] = base_scores
    recs["total_score"] = total_scores
    recs["match_percent"] = match_pct.round(1)

    df_like, dir_map, cast_map, genre_map = build_user_like_metadata(
        user_id, ratings_df, movies
    )

    genre_counts = Counter()
    for gs in df_like.get("genres_clean", []):
        if isinstance(gs, list):
            genre_counts.update(gs)
    favorite_genres = [g for g, _ in genre_counts.most_common(4)]

    liked_rows = df_like.dropna(subset=["movie_idx"]).copy()
    liked_rows["movie_idx"] = liked_rows["movie_idx"].astype(int)

    liked_idxs = liked_rows["movie_idx"].tolist()
    liked_title_by_idx = liked_rows.set_index("movie_idx")["title"].to_dict()
    if "Rating" in liked_rows.columns:
        liked_rating_by_idx = liked_rows.set_index("movie_idx")["Rating"].to_dict()
    else:
        liked_rating_by_idx = {}

    all_reasons = []
    for _, row in recs.iterrows():
        content_reasons = explain_match(
            row,
            dir_map,
            cast_map,
            genre_map,
            favorite_genres,
            liked_idxs,
            liked_title_by_idx,
            liked_rating_by_idx,
            X,
        )
        friend_reasons = []
        if user_friends is not None:
            friend_reasons = friend_reasons_for_movie(
                user_id,
                int(row["movie_idx"]),
                ratings_df,
                friends_lookup,
            )
        reasons_raw = content_reasons + friend_reasons
        reasons = clean_and_pad_reasons(reasons_raw)
        all_reasons.append(reasons)

    recs["reasons"] = all_reasons
    return recs


In [31]:
def interactive_recommender(user_id, max_recs=5):
    """
    Letterboxd-style recommendation popup:
      - shows one recommendation at a time
      - buttons:
           Back
           Yes, I like this recommendation
           Next suggestion
           I've watched this already
      - Only the 'Next suggestion' button moves forward
    """

    recs = recommend_for_user_with_explanations(
        user_id=user_id,
        ratings_df=lb_ratings_all,
        X=X,
        movies=movies,
        user_friends=user_friends_list,
        top_n=max_recs,
    )

    if recs.empty:
        display(Markdown(f"⚠️ No recommendations could be generated for **{user_id}**."))
        return

    recs = recs.reset_index(drop=True)

    state = {"i": 0}  
    out = Output()

    btn_back = Button(
        description="⬅ Back",
        layout=Layout(width="20%", height="40px")
    )

    btn_yes = Button(
        description="Yes",
        layout=Layout(width="30%", height="40px")
    )
    btn_no = Button(
        description="Next suggestion",
        layout=Layout(width="30%", height="40px")
    )
    btn_seen = Button(
        description="I've watched this",
        layout=Layout(width="100%", height="40px")
    )

    btn_back.style.button_color = "#FFFFFF"
    btn_yes.style.button_color = "#FFFFFF"
    btn_no.style.button_color = "#FFFFFF"

    btn_seen.style.button_color = "#1DB954"

    def short_synopsis(row, max_words=40):
        text = row.get("overview_clean", "")
        if not isinstance(text, str):
            return ""
        text = text.strip()
        if not text:
            return ""
        words = text.split()
        if len(words) <= max_words:
            return text[0].upper() + text[1:]
        snippet = " ".join(words[:max_words])
        return snippet[0].upper() + snippet[1:] + "…"

    def render_current():
        """Render the current recommendation card."""
        out.clear_output()
        i = state["i"]

        btn_back.disabled = (i == 0)

        if i >= len(recs):
            with out:
                display(Markdown(
                    "That's all for now — check back later for more recommendations."
                ))
            return

        row = recs.iloc[i]
        title = row["title"]
        year = int(row["year"]) if not pd.isna(row["year"]) else ""
        match = row["match_percent"]
        synopsis = short_synopsis(row)
        reasons = row.get("reasons", []) or ["Similar to your overall taste profile."]

        with out:
            md = f"### 🎬 {title} ({year}) — match: **{match}%**\n\n"
            if synopsis:
                md += f"**Synopsis:** {synopsis}\n\n"
            md += "**Why we picked this for you:**\n"
            md += "\n".join(f"- {r}" for r in reasons)
            display(Markdown(md))

    def on_back(b):
        if state["i"] > 0:
            state["i"] -= 1
            render_current()

    def on_yes(b):
        i = state["i"]
        row = recs.iloc[i]
        title = row["title"]
        out.clear_output()
        with out:
            display(Markdown(
                f"**{title}** added to your watchlist.\n\n"
                "Tap **Next suggestion** to see another recommendation, "
                "or **Back** to revisit the previous one."
            ))

    def on_no(b):
        """Move to the next suggestion."""
        if state["i"] < len(recs) - 1:
            state["i"] += 1
            render_current()
        else:
            out.clear_output()
            with out:
                display(Markdown(
                    "Come back later for more recommendations."
                ))

    def on_seen(b):
        """User has already watched this film; ask about rating."""
        i = state["i"]
        row = recs.iloc[i]
        title = row["title"]
        out.clear_output()
        with out:
            display(Markdown(
                f"You've already watched **{title}**.\n\n"
                "In a real Letterboxd integration, this would jump to the rating screen.\n"
                "Here, tap **Next suggestion** to see another movie, or **Back** to revisit the previous one."
            ))

    btn_back.on_click(on_back)
    btn_yes.on_click(on_yes)
    btn_no.on_click(on_no)
    btn_seen.on_click(on_seen)

    render_current()

    ui = VBox([
        Label(f"Recommendations for {user_id}"),
        HBox([btn_back, btn_yes, btn_no]),
        btn_seen,
        out,
    ])

    display(ui)


In [34]:
interactive_recommender("tharaaaaa", max_recs=6)
